In [1]:
## LOAD DATA FROM MYSQL
import pandas as pd
import numpy as np
import os
import sys
from sqlalchemy import create_engine, text

sys.path.insert(0, os.path.abspath('..'))
import config

engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}"
)

# Load full daily_prices table — sorted by ticker then date
df = pd.read_sql(
    "SELECT * FROM daily_prices ORDER BY ticker, trade_date",
    engine,
    parse_dates=['trade_date']
)

print(f"Loaded       : {len(df):,} rows | {df['ticker'].nunique()} stocks")
print(f"Date range   : {df['trade_date'].min().date()} → {df['trade_date'].max().date()}")
print(f"Markets      : {df['market'].value_counts().to_dict()}")
print(f"\nColumn dtypes:\n{df.dtypes}")

Loaded       : 289,656 rows | 79 stocks
Date range   : 2010-01-04 → 2024-12-30
Markets      : {'India': 177187, 'US': 112469}

Column dtypes:
price_id               int64
stock_id               int64
ticker                   str
market                   str
trade_date     datetime64[s]
open_price           float64
high_price           float64
low_price            float64
close_price          float64
volume                 int64
dtype: object


In [2]:
## DATA QUALITY CHECK & CLEANING

print("=" * 55)
print("  NULL CHECK")
print("=" * 55)
print(df.isnull().sum())

print("\n" + "=" * 55)
print("  ZERO / NEGATIVE PRICES")
print("=" * 55)
bad_prices = df[df['close_price'] <= 0]
print(f"Rows with close_price <= 0 : {len(bad_prices)}")

print("\n" + "=" * 55)
print("  HISTORY LENGTH PER STOCK")
print("=" * 55)
history = df.groupby(['ticker','market'])['trade_date'].count().reset_index()
history.columns = ['ticker', 'market', 'trading_days']
short = history[history['trading_days'] < 1500]
print(f"Stocks with < 1500 trading days (listed after ~2015):")
print(short.sort_values('trading_days').to_string(index=False))

print("\n" + "=" * 55)
print("  FORWARD-FILL ANY GAPS (within each stock)")
print("=" * 55)
# Sort first — always required before groupby rolling ops
df = df.sort_values(['ticker', 'trade_date']).reset_index(drop=True)

for col in ['open_price', 'high_price', 'low_price', 'close_price', 'volume']:
    df[col] = df.groupby('ticker')[col].ffill()

print(f"Nulls remaining after ffill:\n{df[['open_price','close_price','volume']].isnull().sum()}")

  NULL CHECK
price_id       0
stock_id       0
ticker         0
market         0
trade_date     0
open_price     0
high_price     0
low_price      0
close_price    0
volume         0
dtype: int64

  ZERO / NEGATIVE PRICES
Rows with close_price <= 0 : 0

  HISTORY LENGTH PER STOCK
Stocks with < 1500 trading days (listed after ~2015):
Empty DataFrame
Columns: [ticker, market, trading_days]
Index: []

  FORWARD-FILL ANY GAPS (within each stock)
Nulls remaining after ffill:
open_price     0
close_price    0
volume         0
dtype: int64


In [3]:
## FEATURE ENGINEERING
# All calculations are done per ticker using groupby
# so returns never bleed across stock boundaries

# ── Daily return ─────────────────────────────────────────────
# Simple % change: (close_today - close_yesterday) / close_yesterday
df['daily_return'] = df.groupby('ticker')['close_price'].pct_change()

# ── Log return ───────────────────────────────────────────────
# ln(close_t / close_t-1)
# Preferred for statistical analysis — log returns are additive
# and better approximate a normal distribution than simple returns
df['log_return'] = df.groupby('ticker')['close_price'].transform(
    lambda x: np.log(x / x.shift(1))
)

# ── Moving averages ──────────────────────────────────────────
# MA50  → short-term trend
# MA200 → long-term trend
# First 49 / 199 rows per stock will be NaN — that's expected
df['ma_50']  = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=50, min_periods=50).mean()
)
df['ma_200'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=200, min_periods=200).mean()
)

print("Features engineered successfully!")
print(f"\nNew columns added: daily_return, log_return, ma_50, ma_200")
print(f"\nSample — TCS.NS (rows 200-204 to see populated MAs):")
sample = df[df['ticker'] == 'TCS.NS'][['trade_date','close_price',
           'daily_return','log_return','ma_50','ma_200']].iloc[200:205]
print(sample.to_string(index=False))

Features engineered successfully!

New columns added: daily_return, log_return, ma_50, ma_200

Sample — TCS.NS (rows 200-204 to see populated MAs):
trade_date  close_price  daily_return  log_return      ma_50     ma_200
2010-10-20     340.0016     -0.004650   -0.004661 319.420022 282.567041
2010-10-21     348.1022      0.023825    0.023546 320.323986 283.010912
2010-10-22     367.1628      0.054756    0.053309 321.642340 283.579404
2010-10-25     376.6049      0.025716    0.025391 323.142830 284.230373
2010-10-26     375.0695     -0.004077   -0.004085 324.614730 284.898846


In [4]:
## SAVE RESULTS

# ── Backup to CSV ────────────────────────────────────────────
csv_path = '../data/processed/all_stocks_features.csv'
df.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}  ({os.path.getsize(csv_path)/1e6:.1f} MB)")

# ── Create stock_features table in MySQL ─────────────────────
create_features = """
CREATE TABLE IF NOT EXISTS stock_features (
    feature_id    BIGINT               AUTO_INCREMENT PRIMARY KEY,
    stock_id      INT                  NOT NULL,
    ticker        VARCHAR(20)          NOT NULL,
    market        ENUM('India', 'US')  NOT NULL,
    trade_date    DATE                 NOT NULL,
    close_price   DECIMAL(12, 4),
    daily_return  DECIMAL(10, 6),
    log_return    DECIMAL(10, 6),
    ma_50         DECIMAL(12, 4),
    ma_200        DECIMAL(12, 4),
    FOREIGN KEY   (stock_id) REFERENCES stocks(stock_id),
    UNIQUE KEY    uq_feat_ticker_date (ticker, trade_date),
    INDEX         idx_feat_ticker     (ticker),
    INDEX         idx_feat_market     (market, trade_date)
)
"""

with engine.connect() as conn:
    conn.execute(text(create_features))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE stock_features"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()

# Keep only the columns the table expects
features_df = df[['stock_id', 'ticker', 'market', 'trade_date',
                   'close_price', 'daily_return', 'log_return',
                   'ma_50', 'ma_200']].copy()

features_df.to_sql('stock_features', engine, if_exists='append',
                   index=False, chunksize=500, method='multi')

print(f"Inserted {len(features_df):,} rows into stock_features table")

CSV saved → ../data/processed/all_stocks_features.csv  (42.4 MB)
Inserted 289,656 rows into stock_features table


In [5]:
## VERIFICATION & SUMMARY

print("=" * 55)
print("  MYSQL VERIFICATION")
print("=" * 55)
with engine.connect() as conn:
    r = conn.execute(text("SELECT COUNT(*) FROM stock_features")).fetchone()
    print(f"Rows in stock_features : {r[0]:,}")

    r = conn.execute(text("""
        SELECT market,
               COUNT(DISTINCT ticker)          AS stocks,
               ROUND(AVG(daily_return)*100, 4) AS avg_daily_return_pct,
               ROUND(STD(daily_return)*100, 4) AS avg_volatility_pct
        FROM stock_features
        WHERE daily_return IS NOT NULL
        GROUP BY market
    """))
    print(f"\n{'Market':<8} {'Stocks':>6} {'Avg Daily Ret%':>15} {'Avg Volatility%':>16}")
    print("-" * 48)
    for row in r:
        print(f"{row.market:<8} {row.stocks:>6} {row.avg_daily_return_pct:>15} {row.avg_volatility_pct:>16}")

print("\n" + "=" * 55)
print("  TOP 5 SINGLE-DAY GAINS (sanity check)")
print("=" * 55)
top_gains = df.nlargest(5, 'daily_return')[
    ['ticker','market','trade_date','close_price','daily_return']]
print(top_gains.to_string(index=False))

print("\n" + "=" * 55)
print("  TOP 5 SINGLE-DAY DROPS")
print("=" * 55)
top_drops = df.nsmallest(5, 'daily_return')[
    ['ticker','market','trade_date','close_price','daily_return']]
print(top_drops.to_string(index=False))

  MYSQL VERIFICATION
Rows in stock_features : 289,656

Market   Stocks  Avg Daily Ret%  Avg Volatility%
------------------------------------------------
India        49          0.0823           1.9606
US           30          0.0923           1.9749

  TOP 5 SINGLE-DAY GAINS (sanity check)
       ticker market trade_date  close_price  daily_return
          AMD     US 2016-04-22       3.9900      0.522901
INDUSINDBK.NS  India 2020-03-26     420.9504      0.446731
         NFLX     US 2013-01-24       2.0980      0.422276
         NVDA     US 2016-11-11       2.1606      0.298047
         META     US 2013-07-25      34.0925      0.296115

  TOP 5 SINGLE-DAY DROPS
      ticker market trade_date  close_price  daily_return
NESTLEIND.NS  India 2010-01-08     103.0581     -0.763338
 ADANIENT.NS  India 2015-06-03      56.3192     -0.387493
        NFLX     US 2022-04-20      22.6190     -0.351166
        NFLX     US 2011-10-25       1.1053     -0.348943
 ADANIENT.NS  India 2023-02-01    2131